# Week 3 - Gradient Bandit Algorithms

## Context

Week 1 and Week 2 used action-value methods. Epsilon-greedy, optimistic initial values, and UCB all try to estimate `Q(a)`, the value of each action.

Gradient bandits are different. Instead of learning action values, they learn action preferences `H(a)`. Those preferences are then converted into action probabilities with a softmax policy. This makes Week 3 a useful step from value-based bandit methods toward policy-based reinforcement learning methods.

## Setup

This notebook imports the shared implementation from `src/` rather than redefining classes locally. That keeps the notebook focused on understanding and running experiments while the reusable code remains in the main codebase.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import matplotlib.pyplot as plt
import numpy as np

from src.bandits.agents import GradientBanditAgent
from src.bandits.experiments import run_multiple_agent_experiments
from src.utils.plotting import plot_average_reward, plot_optimal_action_percentage

## Action Preferences And Softmax

In gradient bandits, `H(a)` is a preference score, not a reward estimate. A higher preference means the action becomes more likely to be selected, but it does not say that the reward itself equals that preference.

The policy is formed with softmax:

$$
\pi(a) = \frac{\exp(H(a))}{\sum_b \exp(H(b))}
$$

If all preferences are equal, the action probabilities are uniform. As one preference grows larger than the rest, that action gets selected more often, but the policy remains stochastic rather than becoming immediately greedy.

## Agents To Compare

We will compare four gradient bandit agents. The comparison varies two things:
- whether a reward baseline is used
- whether the learning rate is `0.1` or `0.4`

This lets us examine both stability and learning speed.

In [ ]:
agent_configs = {
    "Gradient alpha=0.1 with baseline": {
        "agent_class": GradientBanditAgent,
        "agent_kwargs": {
            "alpha": 0.1,
            "use_baseline": True,
        },
    },
    "Gradient alpha=0.1 without baseline": {
        "agent_class": GradientBanditAgent,
        "agent_kwargs": {
            "alpha": 0.1,
            "use_baseline": False,
        },
    },
    "Gradient alpha=0.4 with baseline": {
        "agent_class": GradientBanditAgent,
        "agent_kwargs": {
            "alpha": 0.4,
            "use_baseline": True,
        },
    },
    "Gradient alpha=0.4 without baseline": {
        "agent_class": GradientBanditAgent,
        "agent_kwargs": {
            "alpha": 0.4,
            "use_baseline": False,
        },
    },
}

## Bandit Setup

In this experiment, the true action values are centered around `4.0` instead of `0.0`. This follows the Sutton and Barto style gradient bandit setup.

That shift matters because the reward baseline becomes more meaningful when most rewards are positive. Without a baseline, the raw reward alone is a weaker learning signal.

In [ ]:
bandit_kwargs = {
    "true_reward_mean": 4.0,
    "true_reward_std": 1.0,
    "reward_std": 1.0,
}

## Debug Run

Before running the full experiment, it is useful to run a smaller version first. This gives a quick check that the setup is behaving sensibly and provides an early look at the curves before spending time on the larger run.

In [ ]:
results = run_multiple_agent_experiments(
    agent_configs=agent_configs,
    n_runs=200,
    n_steps=1000,
    n_actions=10,
    bandit_kwargs=bandit_kwargs,
)

results.keys()

## Plot The Debug Results

Now we visualize the smaller run. These curves will still be somewhat noisy, but they should already suggest whether the baseline versions are learning more steadily.

In [ ]:
plot_average_reward(
    results,
    title="Gradient Bandit: Average Reward over Time",
)

plot_optimal_action_percentage(
    results,
    title="Gradient Bandit: Optimal Action Selection over Time",
)

## Debug Interpretation

In this shifted-reward setting, the baseline versions should usually look more stable. Without the baseline, the agent reacts to raw positive rewards instead of comparing each reward against what is normally expected. That often makes the update noisier and less informative.

## Final Run

Once the debug run looks reasonable, we scale up to the full experiment with `2000` runs. This produces smoother averages and gives a clearer comparison between the four agents.

In [ ]:
final_results = run_multiple_agent_experiments(
    agent_configs=agent_configs,
    n_runs=2000,
    n_steps=1000,
    n_actions=10,
    bandit_kwargs=bandit_kwargs,
)

results = final_results

## Save Final Plots

These plots are saved into `results/week_03/` so they can be referenced from the README and reviewed outside the notebook.

In [ ]:
plot_average_reward(
    results,
    save_path="../results/week_03/gradient_bandit_average_reward.png",
    title="Gradient Bandit: Average Reward over Time",
)

plot_optimal_action_percentage(
    results,
    save_path="../results/week_03/gradient_bandit_optimal_action.png",
    title="Gradient Bandit: Optimal Action Selection over Time",
)

## Final Interpretation

- Gradient bandits with a baseline should usually perform more stably.
- The baseline helps because it compares reward against the expected average reward instead of using the raw reward alone.
- Without a baseline, positive rewards in the shifted setting can make the update signal less meaningful.
- Gradient bandits directly learn action-selection probabilities rather than action-value estimates.

## Technical Takeaways

- Gradient bandits optimize a policy over actions instead of learning `Q(a)` directly.
- Softmax turns preferences into a valid probability distribution.
- The average-reward baseline reduces variance and makes the update more informative.